Direct Mail Fundraising - Module 4 - Samuel Scott

In [1]:
#!pip install xgboost

In [2]:
#import h2o
#import lime
import matplotlib.pyplot as plt
import mlba 
import numpy as np
import pandas as pd
import xgboost
#from h20.automl import H20AutoML
#from lime.lime_tabular import LimeTabularExplainer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, RandomForestClassifier)
from sklearn.metrics import auc, roc_curve, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

## The Organization
A national veterans' organization is one of the largest direct-mail fundraisers in the United States, with an
in-house database of more than 13 million donors. Like every large fundraiser, it faces a cost-
effectiveness problem: most of the people it mails do not respond. Its recent mailings show an overall
response rate of 5.1 percent. Among those who do respond, the average donation is 13.00 dollars, while
each mailing, which includes personalized address labels and an assortment of cards and envelopes,
costs 0.68 dollars to produce and send.
Those economics are why targeting matters. Mailing everyone returns an expected value of about 0.051
times 13.00 minus 0.68 per piece, which is roughly negative 0.02 dollars, so a blanket campaign loses
money on average. The organization wants a model that finds the names whose expected donations
more than cover the cost of reaching them, so the expected net profit of the campaign is maximized.

## The Data
Fundraising.csv is a sample drawn from the organization's database. It contains 3,120 records, one row
per donor, with 20 predictor variables plus two outcome variables. The predictors fall into three groups:
donor and household characteristics (home ownership, number of children, household income, gender,
and a 0 to 9 wealth rating), neighborhood economics (average home value, median and average
neighborhood income, and the percent of the neighborhood earning under 15K), and giving history
(lifetime number of promotions received, lifetime gift total, largest gift, most recent gift, months since the
last donation, the lag between the first and second gift, and the average gift to date). Four zip-code group
dummy variables capture geography. Full definitions for every field are in Table 21.9 below.
The target variable, TARGET_B, is a binary indicator of response: 1 for donors and 0 for non-donors. A
second outcome, TARGET_D, records the donation amount but is not used in this case. To support
modeling on a rare event, the sample is deliberately oversampled to a 50/50 split of donors and non-
donors, even though donors are only about 5.1 percent of the real population. You will need to undo this
oversampling when you translate model results into real-world net profit.

### Question 1 - Data Preparation
Load Fundraising.csv and prepare it for predictive analysis. Partition the data into 60 percent training and
40 percent validation, setting the random seed to 12345 so your split is reproducible.

In [3]:
df = pd.read_csv('C:/Users/samsc/Desktop/ADS-505/Fundraising.csv')
df.head()

,Row Id,Row Id.,zipconvert_2,zipconvert_3,zipconvert_4,zipconvert_5,homeowner dummy,NUMCHLD,INCOME,gender dummy,...,IC15,NUMPROM,RAMNTALL,MAXRAMNT,LASTGIFT,totalmonths,TIMELAG,AVGGIFT,TARGET_B,TARGET_D
0,1,17,0,1,0,0,1,1,5,1,...,1,74,102.0,6.0,5.0,29,3,4.857143,1,5.0
1,2,25,1,0,0,0,1,1,1,0,...,4,46,94.0,12.0,12.0,34,6,9.400000,1,10.0
2,3,29,0,0,0,1,0,2,5,1,...,13,32,30.0,10.0,5.0,29,7,4.285714,1,5.0
3,4,38,0,0,0,1,1,1,3,0,...,4,94,177.0,10.0,8.0,30,3,7.080000,0,0.0
4,5,40,0,1,0,0,1,1,4,0,...,7,20,23.0,11.0,11.0,30,6,7.666667,0,0.0


In [4]:
df.info()
df['TARGET_B'].value_counts() # check for the 50/50 data donor distribution so no class imbalance

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3120 entries, 0 to 3119
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Row Id           3120 non-null   int64  
 1   Row Id.          3120 non-null   int64  
 2   zipconvert_2     3120 non-null   int64  
 3   zipconvert_3     3120 non-null   int64  
 4   zipconvert_4     3120 non-null   int64  
 5   zipconvert_5     3120 non-null   int64  
 6   homeowner dummy  3120 non-null   int64  
 7   NUMCHLD          3120 non-null   int64  
 8   INCOME           3120 non-null   int64  
 9   gender dummy     3120 non-null   int64  
 10  WEALTH           3120 non-null   int64  
 11  HV               3120 non-null   int64  
 12  Icmed            3120 non-null   int64  
 13  Icavg            3120 non-null   int64  
 14  IC15             3120 non-null   int64  
 15  NUMPROM          3120 non-null   int64  
 16  RAMNTALL         3120 non-null   float64
 17  MAXRAMNT      

TARGET_B
1    1560
0    1560
Name: count, dtype: int64

In [5]:
predictors = df.drop(columns=['Row Id','Row Id.', 'TARGET_D', 'TARGET_B'])
predictors = predictors.columns
outcome = 'TARGET_B'

X = df[predictors]
y = df[outcome]

X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.4, random_state=12345)

### Question 2 - Build, Evaluate, and Select a Model
Work through the following steps to build several models, evaluate them in business terms, and choose
one.

#### 2.1 Select Classification Methods
Run at least two classification models of your choosing. Suitable candidates for this case include logistic
regression, classification trees, bagging, AdaBoost, random forest, linear discriminant analysis, and
neural networks. Do not use TARGET_D as a predictor. Describe each model in enough detail (method,
parameters, predictors used) that another analyst could replicate it

In [6]:
#LDA
lda = LinearDiscriminantAnalysis() #think about what parameters could optimize model?
lda.fit(X_train, y_train)

#Accuracy
y_pred_lda = lda.predict(X_holdout)
lda_accuracy = round((y_pred_lda == y_holdout).mean(), 4)

#Confusion Matrix
lda_cm = confusion_matrix(y_holdout, y_pred_lda)
lda_cm #what attribute is most harmful to business? 

#Calculate Precison Recall and F1-Score
lda_precision = round(precision_score(y_holdout, y_pred_lda), 4)
lda_recall = round(recall_score(y_holdout, y_pred_lda), 4)
lda_f1 = round(f1_score(y_holdout, y_pred_lda), 4)
lda_results = pd.DataFrame({
                'Lda Acc: ' : lda_accuracy,
                'Lda Precision: ': lda_precision,
                'Lda Recall: ': lda_recall,
                'Lda F1_Score: ': lda_f1,
}, index=['LDA'])
lda_results

,Lda Acc:,Lda Precision:,Lda Recall:,Lda F1_Score:
LDA,0.5753,0.561,0.6033,0.5814


False positives are harmful because we would be mailing to someone who is not donating, which LDA does a good job at since there are 0 false positives. The harmful attribute are false negatives because we would be predicting not donor but actually is a donor. If this donor gave a lot of money and we fail to mail them, it would be a huge profit loss. Since this is the case, recall is the statistic of most importance for our business and the LDA score is 0.6033. 

Describe each model in enough detail (method, parameters, predictors used) that another analyst could replicate it:
The LDA model used a majority of the predictors in the data set. The predictors removed were TARGET_D, which is a numerical outcome variable
which LDA does not specialize in since it is a classifier not a regressor. Row Id and Row Id. were removed since there are IDs and do not contribute to predicting our TARGET_B variable, which is classifying whether a person will donate or not. Obviously the TARGET_B variables was removed because that is our outcome variable for the LDA model. Priors parameter was used because the donors 5.1% of the population are donors so priors sets the LDA model to realistically account for the real business data, the data was set to have 50/50 donors and non-donors so this change is necessary. LDA is used because it is a classifier and it is great at distinguishing binary outcome variables problems.   

In [7]:
#AdaBoost
adab = AdaBoostClassifier(n_estimators=1000, random_state=12345)
adab.fit(X_train, y_train)
y_pred_adab = adab.predict(X_holdout)

adab_accuracy = round((y_pred_adab == y_holdout).mean(), 4)

adab_cm = confusion_matrix(y_holdout, y_pred_adab)
adab_cm

#Calculate Precison Recall and F1-Score
adab_precision = round(precision_score(y_holdout, y_pred_adab), 4)
adab_recall = round(recall_score(y_holdout, y_pred_adab), 4)
adab_f1 = round(f1_score(y_holdout, y_pred_adab), 4)
adab_results = pd.DataFrame({
                'Ada Acc: ' : adab_accuracy,
                'Ada Precision: ': adab_precision,
                'Ada Recall: ': adab_recall,
                'Ada F1_Score: ': adab_f1,
}, index=['AdaBoost'])
adab_results

,Ada Acc:,Ada Precision:,Ada Recall:,Ada F1_Score:
AdaBoost,0.5481,0.5373,0.5426,0.54


In [8]:
#Bagging
bag = BaggingClassifier(n_estimators=1000, n_jobs=-1, random_state=12345)
bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_holdout)

bag_accuracy = round((y_pred_bag == y_holdout).mean(), 4)

bag_cm = confusion_matrix(y_holdout, y_pred_bag)
bag_cm

#Calculate Precison Recall and F1-Score
bag_precision = round(precision_score(y_holdout, y_pred_bag), 4)
bag_recall = round(recall_score(y_holdout, y_pred_bag), 4)
bag_f1 = round(f1_score(y_holdout, y_pred_bag), 4)
bag_results = pd.DataFrame({
                'Bag Acc: ' : bag_accuracy,
                'Bag Precision: ': bag_precision,
                'Bag Recall: ': bag_recall,
                'Bag F1_Score: ': bag_f1,
}, index=['Bag'])
bag_results

,Bag Acc:,Bag Precision:,Bag Recall:,Bag F1_Score:
Bag,0.5593,0.5485,0.5557,0.5521


These metrics are important, but I realize that the business wants to find people that are most likely to donate so using proba would help determine that for the DirectMailFundraising. 

#### 2.2 Reason About the Weighted Sample
Explain the reasoning behind training on a 50/50 sample of donors and non-donors. Why not simply draw
a random sample from the original database, where donors are only about 5.1 percent of the records?

In [21]:
#Training 
y_proba_lda_train = lda.predict_proba(X_train)[:, 1]  #get the probabilities of just the donors using :, 1 to get second column which has donors=1
y_proba_adab_train = adab.predict_proba(X_train)[:, 1]
y_proba_bag_train = bag.predict_proba(X_train)[:, 1]

lda_proba_results_train = pd.DataFrame({
                    'actual' : y_train,
                    'probabilities': y_proba_lda_train
})
print(lda_proba_results_train.head())

adab_proba_results_train = pd.DataFrame({
                    'actual' : y_holdout,
                    'probabilities': y_proba_adab_train
})
print(adab_proba_results_train.head())

bag_proba_results_train = pd.DataFrame({
                    'actual' : y_holdout,
                    'probabilities': y_proba_bag_train
})
print(bag_proba_results_train.head())

,actual,probabilities
957,0,0.487800
1696,1,0.530409
981,1,0.519656
1438,0,0.574955
2039,1,0.496041
...,...,...
1158,0,0.504639
581,1,0.479895
2958,0,0.487561
1125,1,0.467921


The reasoning behind training of a 50/50 sample of donors and non-donors because we can have a model that can learn and determine which records are going to be donors or non-donors. The original data has only 5.1% of the data having records where donors exist, due to this there is a class imbalance. Simply drawing a random sample it is unlikely to draw a record that is a donor, and because of this the model would not be able to determine donor records very well or at all. 

#### 2.3 Calculate Net Profit
For each method, calculate the cumulative gains of net profit for both the training and validation sets,
based on the actual 5.1 percent response rate. Recall that the expected donation given a donor is 13.00
dollars and the total cost of each mailing is 0.68 dollars.

In [25]:
#Current business model
response_rate = 0.051 #5.1%
avg_donation = 13.00
avg_mailing = 0.68
adj_pop_rate = response_rate / 0.5
current_earnings = response_rate * avg_donation - 0.68   
#print(adj_pop_rate)
current_earnings #(roughly losing 2 cents)

-0.017000000000000126

In [ ]:
#Cumulative Gains Net Profit(Training):



In [40]:
#create a predicted_spending column by creating a mask. All donation probabilities that actually donate

#old strategy (the problem with this strategy is that it is all or nothing. If probability is below 0.50 there is a 0% chance which does not capture
#the probabilities correctly for cases less than 0.5 probability:

#lda_mask_1 = lda_proba_results[lda_proba_results['probabilities'] > 0.5] # make a mask based on how likelihood they are to donate
#lda_proba_results['Predicted_Spending'] = 0.00  #impute all dataframe column with 0.00 dollars
#lda_proba_results.loc[lda_mask_1.index, 'Predicted_Spending'] = avg_donation #if probability is > 0.5 fill with avg_donation amount

#lda computed spending:

#lda_proba_results['Expected_Spending'] = (lda_proba_results['probabilities'] * adj_pop_rate * lda_proba_results['Predicted_Spending']).round(2)
#lda_proba_results


#new strategy:
lda_proba_results['Expected_Spending'] = (lda_proba_results['probabilities'] * adj_pop_rate * avg_donation).round(2)
lda_profit_model = (lda_proba_results['Expected_Spending'] - avg_mailing).sum()
lda_profit_model

np.float64(-18.01000000000006)

#### 2.4 Draw Cumulative Gains Curves
Plot the validation-set net profit cumulative gains curves for your models on a single chart, with net profit
on the y-axis and the proportion of the list mailed on the x-axis. Is there a model that dominates the
others? If the curves cross, note that the best model then depends on how far down the list the
organization intends to mail

#### 2.5 Select the Best Model
Based on your cumulative gains analysis, which model is best for this campaign, and why? Ground your
choice in validation-set net profit rather than overall accuracy alone.

#### Question 3 - Score the Future Candidates
FutureFundraising.csv holds the attributes for future mailing candidates. Using your best model from
Question 2, predict which candidates are donors and which are non-donors, and list them in descending
order of their probability of being a donor. Starting at the top of this ranked list, roughly how far down
would you mail, and what is the reasoning behind that cutoff?